# Example Usage of Metics in TopicGPT: 20 Newsgroups Dataset

In this notebook, we will use the 20 Newsgroups dataset to demonstrate the use of the topicgpt package

In [1]:
import sys
import os

# Add the 'src' directory to the Python path
sys.path.append(os.path.abspath("../src"))

### Configurations

TRAIN = False
PROVIDER = "openai"  # "openai" or "anthropic" or "gemini"
EMBEDDING_PATH = f"./SavedEmbeddings/{PROVIDER}_news.pkl"

In [2]:
# select your own API key here. (Note: This specific code will not work for you unless you specified an environment variable for OPENAI_API_KEY)
import os
from dotenv import load_dotenv

load_dotenv()

if PROVIDER == "openai":
    api_key = os.environ.get('OPENAI_API_KEY')
    prompting_model = "gpt-3.5-turbo-16k"
    embedding_model = "text-embedding-ada-002"
elif PROVIDER == "gemini":
    api_key = os.environ.get('GEMINI_API_KEY')
    prompting_model = "gemini-2.0-flash-lite"
    embedding_model = "gemini-embedding-001"
elif PROVIDER == "anthropic":
    api_key = os.environ.get('ANTHROPIC_API_KEY')
    prompting_model = "claude-sonnet-4-20250514"
    embedding_model = "all-MiniLM-L6-v2"

### Load Data

In [3]:
# from sklearn.datasets import fetch_20newsgroups

# data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes')) #download the 20 Newsgroups dataset
# corpus = data['data']
# corpus = [doc for doc in corpus if doc != ""]

In [9]:
from sklearn.datasets import fetch_20newsgroups

data = fetch_20newsgroups(subset='all', remove=('headers', 'footers', 'quotes'))
corpus = data['data']
topic_names = data.target_names  # e.g., ['alt.atheism', 'comp.graphics', ...]

filtered_corpus = []
filtered_labels = []

print(f'Number of docs before filtering: {len(corpus)}')

for doc, label in zip(corpus, data.target):
    if doc != " ":
        filtered_corpus.append(doc)
        filtered_labels.append(label)

corpus = filtered_corpus
print(f'Number of docs after filtering: {len(corpus)}')
# Now, filtered_corpus contains only non-empty docs,
# and filtered_labels contains their corresponding ground-truth labels.

Number of docs before filtering: 18846
Number of docs after filtering: 18837


## Initialize and fit the model 

In [59]:
from topicgpt.TopicGPT import TopicGPT
if TRAIN:
    tm = TopicGPT(
        prompting_model=prompting_model,
        api_key=api_key,
        n_topics=20,  # select 20 topics since the true number of topics is 20
        embedding_model=embedding_model, 
        use_saved_embeddings=False,  # set to False to train the model from scratch
    )
    tm.fit(corpus)  # train the model on the corpus
    tm.save_embeddings(EMBEDDING_PATH) #save the embeddings for future use
else:
    tm = TopicGPT(
        prompting_model=prompting_model,
        embedding_model=embedding_model, 
        api_key=api_key,
        n_topics=20,  # select 20 topics since the true number of topics is 20
        path_saved_embeddings=EMBEDDING_PATH,
        use_saved_embeddings=True,  # set to True to use saved embeddings
    )

In [60]:
tm

TopicGPT object with the following parameters:
------------------------------------------------------------------------------------------------------------------------------------------------------
n_topics: 20
prompting_model: gpt-3.5-turbo-16k
max_number_of_tokens: 16384
corpus_instruction: 
embedding_model: text-embedding-ada-002
clusterer: <topicgpt.Clustering.Clustering_and_DimRed object at 0x000001FB4DC048F0>
n_topwords: 2000
n_topwords_description: 500
topword_extraction_methods: ['tfidf', 'cosine_similarity']
compute_vocab_hyperparams: {'verbose': True}
enhancer: TopwordEnhancement(model = gpt-3.5-turbo-16k)
topic_prompting: <topicgpt.TopicPrompting.TopicPrompting object at 0x000001FB4DD45A60>

## Get an overview over the identified topics

In [61]:
### Some information about the trained model
print(f'Number of documents: {len(corpus)}')
print(f'document embedding shape: {tm.document_embeddings.shape}')
print(f'number of vocab: {len(list(tm.vocab_embeddings.keys()))}')
print(f'vocab embedding shape: {list(tm.vocab_embeddings.values())[0].shape}') #shape of a single vocab embedding

Number of documents: 18466
document embedding shape: (18466, 1536)
number of vocab: 21143
vocab embedding shape: (1536,)


In [62]:
# We need the list of topics to compute the metrics which is done extract_topics()
tm.extract_topics(corpus)

Processing corpus: 100%|██████████| 18466/18466 [00:13<00:00, 1324.68it/s]
d:\Research\GPTopic\topicgpt-venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Most common words in the vocabulary:
n't: 17116
would: 10872
one: 10144
people: 6442
like: 6409
get: 5815
know: 5742
also: 5588
use: 4987
think: 4982
UMAP(angular_rp_forest=True, metric='cosine', min_dist=0, n_components=5, n_jobs=1, random_state=42, verbose=True)
Tue Nov 18 12:13:49 2025 Construct fuzzy simplicial set
Tue Nov 18 12:13:49 2025 Finding Nearest Neighbors
Tue Nov 18 12:13:49 2025 Building RP forest with 12 trees
Tue Nov 18 12:13:49 2025 NN descent for 14 iterations
	 1  /  14
	 2  /  14
	 3  /  14
	 4  /  14
	 5  /  14
	 6  /  14
	Stopping threshold met -- exiting after 6 iterations
Tue Nov 18 12:13:52 2025 Finished Nearest Neighbor Search
Tue Nov 18 12:13:52 2025 Construct embedding


Epochs completed:   0%|            0/200 [00:00]

	completed  0  /  200 epochs
	completed  20  /  200 epochs
	completed  40  /  200 epochs
	completed  60  /  200 epochs
	completed  80  /  200 epochs
	completed  100  /  200 epochs
	completed  120  /  200 epochs
	completed  140  /  200 epochs
	completed  160  /  200 epochs
	completed  180  /  200 epochs
Tue Nov 18 12:13:59 2025 Finished embedding


d:\Research\GPTopic\topicgpt-venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
d:\Research\GPTopic\topicgpt-venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


...

	completed  0  /  100 epochs
	completed  10  /  100 epochs
	completed  20  /  100 epochs
	completed  30  /  100 epochs
	completed  40  /  100 epochs
	completed  50  /  100 epochs
	completed  60  /  100 epochs
	completed  70  /  100 epochs
	completed  80  /  100 epochs
	completed  90  /  100 epochs


Computing word-topic matrix: 100%|██████████| 20/20 [00:16<00:00,  1.22it/s]


Epochs completed:   0%|            0/30 [00:00]

	completed  0  /  30 epochs
	completed  3  /  30 epochs
	completed  6  /  30 epochs
	completed  9  /  30 epochs
	completed  12  /  30 epochs
	completed  15  /  30 epochs
	completed  18  /  30 epochs
	completed  21  /  30 epochs
	completed  24  /  30 epochs
	completed  27  /  30 epochs


[Topic: 0,
 Topic: 1,
 Topic: 2,
 Topic: 3,
 Topic: 4,
 Topic: 5,
 Topic: 6,
 Topic: 7,
 Topic: 8,
 Topic: 9,
 Topic: 10,
 Topic: 11,
 Topic: 12,
 Topic: 13,
 Topic: 14,
 Topic: 15,
 Topic: 16,
 Topic: 17,
 Topic: 18]

In [63]:
tm.score()

Scoring 19 topics
Generating descriptions for topics using tm.describe_topics...


100%|██████████| 19/19 [00:14<00:00,  1.35it/s]


TypeError: ADC.score() got an unexpected keyword argument 'new_embeddings'

In [ ]:
# import numpy as np
# import re
# from typing import List, Optional, Any
# from sklearn.metrics.pairwise import cosine_similarity
# from topicgpt.TopicRepresentation import Topic

# class ADS_Intruder:
#     """
#     Intruder-based Average Description Similarity (ADS-Intruder) metric for topic models.

#     This metric measures the average cosine similarity (scaled to [0, 1]) between the documents in a cluster
#     and the descriptions of all other clusters ("intruder" descriptions). Lower values indicate better topic separation,
#     as documents in a cluster are less similar to descriptions of other topics.

#     - Range: 0 to 1 (lower is better)
#     - Use: To evaluate how well-separated the topics are from each other based on their descriptions.
#     """

#     def __init__(self, n_docs: int = -1, embedder: Optional[Any] = None):
#         """
#         Args:
#             n_docs (int): Number of documents per topic to use (-1 means all).
#             embedder: An object with .encode() method for embedding texts.
#         """
#         self.n_docs = n_docs
#         self.embedder = embedder

#     def get_info(self) -> dict:
#         """
#         Get information about the metric.
#         """
#         info = {
#             "metric_name": "Intruder-based Average Description Similarity (ADS-Intruder)",
#             "n_docs": self.n_docs,
#             "metric_range": "-1 to 1, lower is better",
#             "description": "Average cosine similarity between cluster documents and intruder topic descriptions. Lower scores indicate better topic separation.",
#         }
#         return info

#     def score(self, topics: List[Any]) -> float:
#         """
#         Args:
#             topics: List of Topic-like objects, each with:
#                 - topic_desc (str): topic description
#                 - document_embeddings_hd (np.ndarray): document embeddings
#                 - centroid_hd (optional, np.ndarray): cluster centroid
#         Returns:
#             float: The average intruder ADS score across all topics.
#         """
#         assert isinstance(topics, (list, tuple)), "topics must be a list or tuple of Topic objects"
#         assert self.embedder is not None, "embedder must be provided"

#         # Clean the descriptions and embed them
#         descriptions = [self.clean_description(getattr(t, "topic_desc", "")) for t in topics]
#         topic_desc_embeddings = self.embedder.encode(descriptions, convert_to_numpy=True)

#         # Prepare document embeddings for each cluster
#         emb_clusters: List[np.ndarray] = []
#         for t in topics:
#             emb = getattr(t, "document_embeddings_hd", None)
#             assert emb is not None, "Each topic must have 'document_embeddings_hd'"
#             arr = np.atleast_2d(np.asarray(emb))
            
#             # If n_docs > 0, select the most representative documents
#             if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
#                 n_available = arr.shape[0]
#                 n_select = min(self.n_docs, n_available)
#                 centroid = getattr(t, "centroid_hd", None)
#                 if centroid is None:
#                     centroid = arr.mean(axis=0)
#                 else:
#                     centroid = np.asarray(centroid).reshape(-1)
#                 sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
#                 top_idx = np.argsort(sims)[-n_select:][::-1]
#                 arr = arr[top_idx]
#             emb_clusters.append(arr)

#         # Calculate intruder similarities for each cluster
#         cluster_scores: List[float] = []
        
#         for i, cluster_docs in enumerate(emb_clusters):
#             if cluster_docs.size == 0:
#                 cluster_scores.append(np.nan)
#                 continue
            
#             intruder_similarities: List[float] = []
            
#             # Compare with all other clusters' descriptions (intruders)
#             for j, intruder_desc_emb in enumerate(topic_desc_embeddings):
#                 if j == i:  # Skip own description
#                     continue
                
#                 # Calculate similarity between intruder description and cluster documents
#                 intruder_desc = intruder_desc_emb.reshape(1, -1)
#                 sims = cosine_similarity(intruder_desc, cluster_docs)  # Shape: (1, n_docs)
#                 sims = (sims + 1) / 2  # Scale similarity to [0, 1] 
#                 avg_sim = np.mean(sims)  # Average similarity for this intruder
#                 intruder_similarities.append(avg_sim)
            
#             # Average across all intruder descriptions for this cluster
#             if intruder_similarities:
#                 cluster_scores.append(np.mean(intruder_similarities))
#             else:
#                 cluster_scores.append(np.nan)

#         # Final average across all clusters
#         if len(cluster_scores) == 0 or np.all(np.isnan(cluster_scores)):
#             return np.nan
        
#         return round(np.nanmean(cluster_scores), 4)

#     @staticmethod
#     def clean_description(desc: str) -> str:
#         if not desc:
#             return ""
#         desc = desc.strip().strip('\'"')
#         desc = re.sub(r'(\*\*|\*|`)+', '', desc)
#         desc = re.sub(r'^#{1,6}\s*', '', desc, flags=re.MULTILINE)
#         desc = re.sub(r'^\s*[\-\*\+]\s+', '', desc, flags=re.MULTILINE)
#         desc = re.sub(r'^\s*\d+\.\s+', '', desc, flags=re.MULTILINE)
#         desc = re.sub(r'\s+', ' ', desc)
#         return desc.strip()

# import numpy as np
# import re
# from typing import List, Optional, Any
# from sklearn.metrics.pairwise import cosine_similarity

# class ADS:
#     """
#     Average Description Similarity (ADS) metric for topic models.

#     This metric measures the average cosine similarity (scaled to [0, 1]) between the embedding of each document
#     and the embedding of its assigned topic description. Higher values indicate that documents are more similar
#     to their topic description, reflecting better topic coherence.

#     - Range: 0 to 1 (higher is better)
#     - Use: To evaluate how well topic descriptions represent their assigned documents.
#     """

#     def __init__(self, n_docs: int = -1, embedder: Optional[Any] = None):
#         """
#         Args:
#             n_docs (int): Number of documents per topic to use (-1 means all).
#             embedder: An object with a .get_embeddings(list_of_texts) method returning a dict with 'embeddings' key.
#         """
#         self.n_docs = n_docs
#         self.embedder = embedder

#     def get_info(self) -> dict:
#         """
#         Get information about the metric.
#         """
#         info = {
#             "metric_name": "Average Description Similarity (ADS)",
#             "n_docs": self.n_docs,
#             "metric_range": "0 to 1, higher is better",
#             "description": "The average cosine similarity (scaled to [0,1]) between the embedding of each document and the embedding of its topic description.",
#         }
#         return info

#     def score(self, topics: List[Any]) -> float:
#         """
#         Args:
#             topics: List of Topic-like objects, each with:
#                 - topic_description (str)
#                 - document_embeddings_hd (np.ndarray)
#                 - centroid_hd (optional, np.ndarray)
#         Returns:
#             float: The average ADS score across all topics.
#         """
#         assert isinstance(topics, (list, tuple)), "topics must be a list or tuple of Topic objects"
#         assert self.embedder is not None, "embedder must be provided"

#         # Clean the descriptions
#         descriptions = [self.clean_description(getattr(t, "topic_description", "")) for t in topics]

#         # Embed the topic descriptions
#         # topic_desc_embeddings = self.embedder.get_embeddings(descriptions)["embeddings"]
#         topic_desc_embeddings = self.embedder.encode(descriptions, convert_to_numpy=True)

#         # Embed the documents in each topic
#         emb_clusters: List[np.ndarray] = []
#         for t in topics:
#             emb = getattr(t, "document_embeddings_hd", None)
#             assert emb is not None, "Each topic must have 'document_embeddings_hd'"
#             arr = np.atleast_2d(np.asarray(emb))
#             # If n_docs > 0, select the most representative documents (top-k by similarity to centroid)
#             if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
#                 n_available = arr.shape[0]
#                 n_select = min(self.n_docs, n_available)
#                 centroid = getattr(t, "centroid_hd", None)
#                 if centroid is None:
#                     centroid = arr.mean(axis=0)
#                 else:
#                     centroid = np.asarray(centroid).reshape(-1)
#                 sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
#                 top_idx = np.argsort(sims)[-n_select:][::-1]
#                 arr = arr[top_idx]
#             emb_clusters.append(arr)

#         similarity_scores: List[float] = []
#         for i in range(len(emb_clusters)):
#             desc = topic_desc_embeddings[i].reshape(1, -1)
#             docs = emb_clusters[i]
#             if docs.size == 0:
#                 similarity_scores.append(np.nan)
#                 continue
#             sims = cosine_similarity(desc, docs)  # Shape: (1, n_docs)
#             sims_01 = (sims + 1) / 2  # Now in [0, 1]
#             similarity_scores.append(np.nanmean(sims_01))  # Average similarity for the topic

#         if len(similarity_scores) == 0 or np.all(np.isnan(similarity_scores)):
#             return np.nan
#         return round(np.nanmean(similarity_scores), 4)

#     @staticmethod
#     def clean_description(desc: str) -> str:
#         if not desc:
#             return ""
#         desc = desc.strip().strip('\'"')
#         desc = re.sub(r'(\*\*|\*|`)+', '', desc)
#         desc = re.sub(r'^#{1,6}\s*', '', desc, flags=re.MULTILINE)
#         desc = re.sub(r'^\s*[\-\*\+]\s+', '', desc, flags=re.MULTILINE)
#         desc = re.sub(r'^\s*\d+\.\s+', '', desc, flags=re.MULTILINE)
#         desc = re.sub(r'\s+', ' ', desc)
#         return desc.strip()
    

# from sklearn.metrics.pairwise import cosine_similarity
# import warnings
# from typing import List, Optional, Any


# class ADC:
#     """
#     Average Document Coherence (ADC) metric for topic models.

#     This metric measures the average cosine similarity (scaled to [0, 1]) between the embeddings of documents in a cluster
#     and randomly selected "intruder" documents from other clusters. Lower values indicate better topic separation,
#     as documents in a cluster are less similar to documents from other clusters.

#     - Range: 0 to 1 (lower is better)
#     - Use: To evaluate how well-separated the clusters are from each other.
#     """

#     def __init__(
#         self,
#         n_docs: int = -1,  # -1 means all docs in the cluster
#         n_intruder_docs: int = 1,
#     ):
#         assert isinstance(n_docs, int), "n_docs must be an integer"
#         assert isinstance(n_intruder_docs, int), "n_intruder_docs must be an integer"
#         self.n_docs = n_docs
#         self.n_intruder_docs = n_intruder_docs

#     def score_one_intr_per_cluster(
#         self,
#         topic_list: List[Topic],
#         random_state: Optional[Any] = None,
#     ) -> np.ndarray:
#         rng = np.random.default_rng(random_state)
#         emb_clusters: List[np.ndarray] = []
#         for t in topic_list:
#             emb = getattr(t, "document_embeddings_hd", None)
#             assert isinstance(emb, np.ndarray), "Topic objects must have document_embeddings_hd field populated as numpy array."
#             arr = np.atleast_2d(np.asarray(emb))
#             if self.n_docs is not None and self.n_docs > 0 and arr.size > 0:
#                 n_available = arr.shape[0]
#                 if n_available > self.n_docs:
#                     centroid = getattr(t, "centroid_hd", None)
#                     if centroid is None:
#                         centroid = arr.mean(axis=0)
#                     else:
#                         centroid = np.asarray(centroid).reshape(-1)
#                     sims = cosine_similarity(centroid.reshape(1, -1), arr).flatten()
#                     top_idx = np.argsort(sims)[-self.n_docs:][::-1]
#                     arr = arr[top_idx]
#             emb_clusters.append(arr)

#         scores: List[float] = []
#         for i, cluster_emb in enumerate(emb_clusters):
#             if cluster_emb.size == 0:
#                 scores.append(np.nan)
#                 continue
#             other = [np.atleast_2d(c) for j, c in enumerate(emb_clusters) if j != i and c.size > 0]
#             if len(other) == 0:
#                 scores.append(np.nan)
#                 continue
#             other_embs = np.vstack(other)
#             intr_idx = int(rng.integers(0, other_embs.shape[0]))
#             intr_embedding = other_embs[intr_idx]
#             sim = cosine_similarity(intr_embedding.reshape(1, -1), cluster_emb)  # (1, n_docs)
#             sim = (sim + 1) / 2  # Scale to [0, 1]
#             scores.append(float(np.mean(sim)))
#         return np.array(scores)

#     def score_per_cluster(self, topic_list: List[Topic]) -> dict:
#         score_lis: List[np.ndarray] = []
#         for _ in range(self.n_intruder_docs):
#             score_per_cluster = self.score_one_intr_per_cluster(
#                 topic_list
#             )
#             score_lis.append(score_per_cluster)
#         res = np.vstack(score_lis).T
#         mean_scores = np.mean(res, axis=1)
#         ntopics = len(topic_list)
#         results: dict = {}
#         for k in range(ntopics):
#             preview = ""
#             t = topic_list[k]
#             if getattr(t, "documents", None):
#                 preview = " - " + str(t.documents[0])[:40].replace("\n", " ").strip()
#             label = f"cluster_{k}{preview}"
#             results[label] = float(np.round(mean_scores[k], 5))
#         return results

#     def score(self, topics: List[Topic]) -> float:
#         scores = list(self.score_per_cluster(topics).values())
#         if all(np.isnan(scores)):
#             warnings.warn("ADC: All clusters returned NaN (no valid intruder comparisons possible).")
#             return np.nan
#         return float(np.nanmean(scores))